# Импорты и зависимости

In [ ]:
!pip install sympy==1.12 --force-reinstall

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.8/567.8 kB 63.4 MB/s eta 0:00:00
  Attempting uninstall: mpmath
    Found existing installation: mpmath 1.3.0
    Uninstalling mpmath-1.3.0:
      Successfully uninstalled mpmath-1.3.0
  Attempting uninstall: sympy
    Found existing installation: sympy 1.13.3
    Uninstalling sympy-1.13.3:
      Successfully uninstalled sympy-1.13.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.8.0+cu126 requires sympy>=1.13.3, but you have sympy 1.12 which is incompatible.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from einops import rearrange
from torchvision.models import resnet18

# Класс модели

In [ ]:
class LPU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)

    def forward(self, x):
        return x + self.dwconv(x)

class NCSSD(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.x_proj = nn.Linear(dim, dim * 2)
        self.c_proj = nn.Linear(dim, dim)
        self.m = nn.Parameter(torch.ones(num_heads, self.head_dim) * 0.1)

    def forward(self, x):
        B, C, H, W = x.shape
        x_flat = rearrange(x, 'b c h w -> b (h w) c')
        xb = self.x_proj(x_flat)
        B_mat = xb[..., :self.dim]
        x_in = xb[..., self.dim:]
        C_mat = self.c_proj(x_flat)

        B_heads = rearrange(B_mat, 'b l (h d) -> b l h d', h=self.num_heads)
        C_heads = rearrange(C_mat, 'b l (h d) -> b l h d', h=self.num_heads)

        B_mat = rearrange(B_heads, 'b l h d -> b l (h d)')
        C_mat = rearrange(C_heads, 'b l h d -> b l (h d)')

        m = self.m.unsqueeze(0).unsqueeze(0)
        weighted_B = B_mat * m.view(1, 1, -1)
        global_H = torch.matmul(weighted_B.transpose(1, 2), x_in)
        y = torch.matmul(C_mat, global_H)

        y = y * (1.0 / torch.sqrt(torch.tensor(self.dim, dtype=torch.float32, device=y.device)))
        return rearrange(y, 'b (h w) c -> b c h w', h=H, w=W)

class VSSDBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.lpu = LPU(dim)
        self.norm1 = nn.LayerNorm(dim)
        self.ncssd = NCSSD(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))

    def forward(self, x):
        x = self.lpu(x)
        x_flat = rearrange(x, 'b c h w -> b (h w) c')
        ncssd_out = self.ncssd(self.norm1(x_flat).transpose(1, 2).reshape(x.shape))
        x = x + F.gelu(ncssd_out)
        x_flat = rearrange(x, 'b c h w -> b (h w) c')
        x = x + self.ffn(self.norm2(x_flat)).transpose(1, 2).reshape(x.shape)
        return x

class VSSD(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Conv2d(3, 32, kernel_size=4, stride=4)
        self.stage1 = VSSDBlock(32)
        self.down1 = nn.Conv2d(32, 64, 3, stride=2, padding=1)
        self.stage2 = VSSDBlock(64)
        self.down2 = nn.Conv2d(64, 128, 3, stride=2, padding=1)
        self.stage3 = nn.Sequential(VSSDBlock(128), VSSDBlock(128))
        self.norm = nn.LayerNorm(128)
        self.head = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.down1(x)
        x = self.stage2(x)
        x = self.down2(x)
        x = self.stage3(x)
        x = rearrange(x, 'b c h w -> b (h w) c')
        x = self.norm(x.mean(dim=1))
        return self.head(x)

# Обучение и сравнение с ResNet18

In [ ]:
resnet_model = resnet18(weights=None)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, 10)

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
vssd_model = VSSD().to(device)
cnn_model = resnet_model.to(device)

criterion = nn.CrossEntropyLoss()
opt_vssd = torch.optim.SGD(vssd_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
opt_cnn = torch.optim.SGD(cnn_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

def test_accuracy(model):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

print("Начинаем обучение VSSD vs ResNet18 на CIFAR-10...\n")

for epoch in range(10):
    loss_vssd = loss_cnn = 0.0
    vssd_model.train()
    cnn_model.train()

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        # VSSD
        opt_vssd.zero_grad()
        outputs = vssd_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(vssd_model.parameters(), 1.0)
        opt_vssd.step()
        loss_vssd += loss.item()

        # ResNet18
        opt_cnn.zero_grad()
        outputs = cnn_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        opt_cnn.step()
        loss_cnn += loss.item()

    acc_vssd = test_accuracy(vssd_model)
    acc_cnn = test_accuracy(cnn_model)

    print(f"Epoch {epoch+1:2d} | VSSD Loss: {loss_vssd/len(trainloader):.4f} | Acc: {acc_vssd:.2f}% "
          f"| ResNet18 Loss: {loss_cnn/len(trainloader):.4f} | Acc: {acc_cnn:.2f}%")

print("\n✅ Сравнение завершено!")
print(f"Итоговая тестовая точность:\n   VSSD      : {acc_vssd:.2f}%\n   ResNet18  : {acc_cnn:.2f}%")

cuda
Начинаем обучение VSSD vs ResNet18 на CIFAR-10...

Epoch  1 | VSSD Loss: 1.7382 | Acc: 47.01% | ResNet18 Loss: 1.9465 | Acc: 48.36%
Epoch  2 | VSSD Loss: 1.3819 | Acc: 52.11% | ResNet18 Loss: 1.2736 | Acc: 56.78%
Epoch  3 | VSSD Loss: 1.2637 | Acc: 55.57% | ResNet18 Loss: 1.0610 | Acc: 64.47%
Epoch  4 | VSSD Loss: 1.1652 | Acc: 58.70% | ResNet18 Loss: 0.9161 | Acc: 66.81%
Epoch  5 | VSSD Loss: 1.1049 | Acc: 60.91% | ResNet18 Loss: 0.8276 | Acc: 67.21%
Epoch  6 | VSSD Loss: 1.0644 | Acc: 59.76% | ResNet18 Loss: 0.7548 | Acc: 68.77%
Epoch  7 | VSSD Loss: 1.0313 | Acc: 61.64% | ResNet18 Loss: 0.6950 | Acc: 69.45%
Epoch  8 | VSSD Loss: 0.9905 | Acc: 62.38% | ResNet18 Loss: 0.6563 | Acc: 69.90%
Epoch  9 | VSSD Loss: 0.9663 | Acc: 64.47% | ResNet18 Loss: 0.6115 | Acc: 73.09%
Epoch 10 | VSSD Loss: 0.9445 | Acc: 65.55% | ResNet18 Loss: 0.5820 | Acc: 71.12%

✅ Сравнение завершено!
Итоговая тестовая точность:
   VSSD      : 65.55%
   ResNet18  : 71.12%


# Итог
В решении задачи классификации соревновались модели VSSD и ResNet18. В результате обучения на 10 эпохах показала себя лучше ResNet18.